# Semana 4: selecciones y datos vs MC (`03_seleccion.ipynb`)

Programa: [`docs/programa_mes1.md`](../../docs/programa_mes1.md), sección *Semana 4*.

**Muestras:** un archivo de datos DoubleMuon 2016 y uno de MC Drell–Yan (DYJetsToLL_M-50) en NanoAODv9 de
CMS Open Data. A diferencia del archivo de 2012, estos traen variables de calidad (identificación, aislamiento,
parámetro de impacto), vértices y, en el MC, *truth information*.

Las rutas van en `configs/datos.yaml` (las pone el supervisor) y los cortes en `configs/seleccion_z.yaml` (los
completas tú). Mientras las rutas estén vacías, el notebook usa el respaldo sintético, que imita las dos muestras.

**Cómo trabajar este notebook.** Completa las celdas marcadas con `# TODO`. Las celdas de **verificación**
(las que terminan en `assert`) ya están escritas: no las borres ni cambies las tolerancias. Si una verificación
falla, intenta corregirlo tú; si sigue fallando, anótalo en `bitacora.md` en *qué no entendí*. Las celdas
*Tu respuesta* se contestan con tus palabras (2 a 5 frases) y se discuten en la reunión.

Antes del commit: *Kernel → Restart & Run All* (todo debe correr de principio a fin) y luego borra las salidas.

In [ ]:
# --- Configuración: funciona igual en Google Colab y en un entorno conda local ---
import os, sys, subprocess
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/RREYESAL/Hands_on_SSIPN"  # <- si trabajas en un fork, pon aquí su URL
DIR_COLAB = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

if EN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "uproot", "awkward", "vector", "hist", "mplhep", "pyyaml", "aiohttp", "requests", "psutil"], check=True)
    if not DIR_COLAB.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(DIR_COLAB)], check=True)
    os.chdir(DIR_COLAB / "notebooks" / "mes1")

# Buscar la raíz del repositorio (la carpeta que contiene src/hepss) y hacer importable hepss
RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hepss").is_dir()), None)
if RAIZ is None:
    raise RuntimeError("No encuentro src/hepss: corre el notebook desde dentro del repositorio")
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
os.chdir(RAIZ)  # rutas relativas (configs/, data/, plots/) respecto a la raíz
Path("plots").mkdir(exist_ok=True)

# Recarga automática de hepss: si editas src/hepss/*.py, no hace falta reiniciar el kernel
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

# Para forzar los datos sintéticos (sin red), descomentar:
# os.environ["HEPSS_SINTETICO"] = "1"
print("Colab:", EN_COLAB, "| raíz del repositorio:", RAIZ)

### Datos: CMS Open Data o respaldo sintético

`hepss.io.abrir_events` intenta leer CMS Open Data por HTTPS (y por XRootD si `fsspec-xrootd` está instalado).
Si no lo consigue en 30 s (sin red, servidor caído, firewall) o si la ruta no está configurada, **genera una vez
un archivo sintético en `data/` con las mismas ramas** y lo usa, avisando con un `UserWarning`. La variable
`ORIGEN` vale `"opendata"` o `"sintetico"`. Usa siempre `etiqueta_cms(ax, ORIGEN)` en tus gráficas: con datos
sintéticos pone *Simulación sintética* en lugar de *CMS Open Data*, para que nunca se confundan.

Las verificaciones están escritas para pasar con ambos orígenes. Las que solo tienen sentido con datos reales
se comprueban solo si `ORIGEN == "opendata"`.

In [ ]:
import numpy as np
import awkward as ak
import pandas as pd
import hist
import vector
import yaml
import matplotlib.pyplot as plt
import mplhep as hep

from hepss.io import abrir_events, zip_muones
from hepss.seleccion import cargar_config, aplicar_seleccion, masa_par
from hepss.graficas import etiqueta_cms

vector.register_awkward()
cfg = cargar_config("configs/seleccion_z.yaml")
rutas = cargar_config("configs/datos.yaml")
print(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

## E4.1 Leer solo las ramas necesarias (desde el YAML)

Completa `ramas` y `ramas_mc` en `configs/seleccion_z.yaml` y vuelve a correr la celda anterior. Lee `ev_d` (datos, solo
`cfg["ramas"]`) y `ev_mc` (MC, `cfg["ramas"] + cfg["ramas_mc"]`) con `entry_stop=cfg["n_eventos"]`, y agrupa los muones
con `zip_muones`. Define `w_mc`, el peso por evento del MC (arreglo de numpy), a partir de `genWeight`.

In [ ]:
arbol_d, ORIGEN_D = abrir_events("datos2016", urls=rutas["datos2016"])
arbol_mc, ORIGEN_MC = abrir_events("mc2016", urls=rutas["mc2016"])
ORIGEN = "opendata" if ORIGEN_D == ORIGEN_MC == "opendata" else "sintetico"

# TODO
ev_d = ...
ev_mc = ...
mu_d = ...
mu_mc = ...
w_mc = ...

In [ ]:
# Verificación E4.1
assert set(ev_d.fields) == set(cfg["ramas"])
assert set(ev_mc.fields) == set(cfg["ramas"] + cfg["ramas_mc"])
for r in ["Muon_tightId", "Muon_pfRelIso04_all", "Muon_dxy", "Muon_dz", "PV_npvsGood"]:
    assert r in cfg["ramas"], f"Falta {r} en el YAML"
assert len(w_mc) == len(ev_mc)

Explica en una frase cada variable: `tightId`, `pfRelIso04_all`, `dxy`, `dz`, `PV_npvsGood`. Mira los valores de `genWeight`: ¿qué tienen de raro y qué haces con ellos?

**Tu respuesta:** _(escribe aquí)_

## E4.2 Cortes en orden y tabla de cortes (*cutflow*)

1. Completa `cortes_muon` en `configs/seleccion_z.yaml` con todos los cortes del programa, en orden.
2. Implementa `hepss.seleccion.masa_par` y `hepss.seleccion.aplicar_seleccion` (la especificación está en el docstring).
3. Aplica la misma función a datos y a MC (con `pesos=w_mc`) y muestra una tabla con los eventos y la eficiencia
   relativa de cada corte en datos y en MC.

In [ ]:
# TODO
sel_d, vivo_d, cf_d, etapas_d = ...
sel_mc, vivo_mc, cf_mc, etapas_mc = ...

In [ ]:
# Verificación E4.2: reproducible desde el YAML y el número de eventos nunca aumenta
_, _, cf_d2, _ = aplicar_seleccion(mu_d, cargar_config("configs/seleccion_z.yaml"))
assert cf_d2["Eventos"].equals(cf_d["Eventos"])
assert cf_d["Eventos"].is_monotonic_decreasing
assert len(cfg["cortes_muon"]) == 6, "El programa pide 6 cortes por muon"
assert len(cf_d) == len(cfg["cortes_muon"]) + 4
assert len(vivo_d) == len(mu_d) and vivo_d.sum() == cf_d["Eventos"].iloc[-1]

¿Por qué la eficiencia relativa de un corte depende del orden? ¿Qué te dice comparar la eficiencia de un corte en datos y en MC Drell–Yan?

**Tu respuesta:** _(escribe aquí)_

## E4.3 Efecto de cada corte en mμμ, pT y η del muon líder

Con `etapas_d`, grafica mμμ (50 a 130 GeV), pT y η del muon líder después de cada corte (tres paneles, escala log, una
curva por etapa), en `plots/cortes_etapas.png`. Para cuantificar, cuenta en cada etapa los eventos en el pico
(80–100 GeV) y en las bandas laterales (60–80 y 100–120 GeV).

In [ ]:
# TODO

¿Qué corte quita más fondo y cuál quita más señal? ¿Cuál discrimina mejor? ¿Hay cortes que casi no hacen nada? ¿Por qué? Responde con números de tu tabla.

**Tu respuesta:** _(escribe aquí)_

## E4.4 Datos vs MC normalizados, con panel de cociente

Histograma de mμμ en datos (`h_d`) y en MC (`h_mc`, con pesos) en la ventana de masa, con
`storage=hist.storage.Weight()`. Normaliza el MC al área de los datos y dibuja datos (con barras de error), MC (con su
banda de incertidumbre estadística) y un panel inferior con el cociente datos/MC y sus barras de error
(`plots/datos_vs_mc_mz.png`). Calcula el cociente medio en el pico (86–96 GeV), `r_pico`, con su error `err_pico`,
y guarda los errores de los datos por bin en `sD`.

Piensa bien cómo se calculan los errores de un histograma con pesos y qué les pasa al normalizar.

In [ ]:
# TODO
h_d = ...
h_mc = ...
sD = ...
r_pico, err_pico = ..., ...

In [ ]:
# Verificación E4.4
print(f"Cociente medio datos/MC en 86–96 GeV: {r_pico:.3f} ± {err_pico:.3f}")
assert abs(r_pico - 1) < 0.05
assert 0 < err_pico < 0.05
assert np.all(np.isfinite(sD)) and np.all(sD[h_d.values() > 0] > 0)

¿Qué información se pierde al normalizar al mismo área? Mira el cociente bin a bin: ¿hay estructura? ¿Qué desviaciones son significativas (más de 2σ) y qué podría causarlas?

**Tu respuesta:** _(escribe aquí)_

## E4.5 *Pile-up*: número de vértices en datos y MC

Compara `PV_npvsGood` (eventos seleccionados) en datos y MC, normalizado, en `plots/npv.png`. Calcula las medias
`media_d` y `media_mc` (esta última con los pesos del MC).

In [ ]:
# TODO
media_d = ...
media_mc = ...

In [ ]:
# Verificación E4.5
print(f"<npv> datos = {media_d:.1f},  <npv> MC = {media_mc:.1f}")
assert abs(media_d - media_mc) > 0.5, "Busca la diferencia: debería haberla"

¿Qué es el *pile-up*? ¿Por qué el MC no lo reproduce? ¿Qué otras variables de tu selección dependen de él? ¿Cómo repesarías el MC?

**Tu respuesta:** _(escribe aquí)_

## E4.6 *Truth matching* en el MC

`Muon_genPartIdx` es, para cada muon reconstruido, el índice de su partícula generada en la colección `GenPart` del
mismo evento (o −1 si no hay). Para los muones seleccionados del MC, calcula `asociado` (máscara irregular: el muon
tiene una partícula generada asociada y es un muon, |pdgId| = 13) y `frac_asociado`, la fracción de muones
seleccionados que cumple eso. Cuidado: hay que proteger los −1 antes de indexar.

In [ ]:
# TODO
asociado = ...
frac_asociado = ...

In [ ]:
# Verificación E4.6
print(f"Muones seleccionados con un muon generado asociado: {frac_asociado:.2%}")
assert frac_asociado > 0.95

¿Qué relación hay entre el signo de `pdgId` y la carga del muon? Compruébalo con tus datos. ¿La fracción asociada es mayor o menor antes de la selección? ¿Por qué?

**Tu respuesta:** _(escribe aquí)_

## E4.7 Resolución de masa en MC: m(reco) − m(gen), barril y endcap

Para los pares con ambos muones asociados, calcula la masa del par con los muones **generados** y la diferencia
`m(reco) − m(gen)`. Haz el histograma para pares en barril y en endcap (regiones de `cfg["regiones_eta"]`), con
pesos, en `plots/resolucion_mz.png`. Estima la anchura de cada uno, `s_b` y `s_e` en GeV. Elige un estimador
que no sea muy sensible a colas y justifícalo.

In [ ]:
# TODO
s_b = ...
s_e = ...

In [ ]:
# Verificación E4.7
print(f"Resolución barril = {s_b:.2f} GeV, endcap = {s_e:.2f} GeV")
assert s_e > s_b
assert 0.1 < s_b < 5, "Revisa unidades y estimador"

¿Por qué m(reco) − m(gen) mide la resolución mejor que la anchura del pico del Z reconstruido? ¿Es coherente con lo que viste en E3.8?

**Tu respuesta:** _(escribe aquí)_

## E4.8 Presentación de 10 minutos

Prepara 4 o 5 diapositivas: datos usados (*datasets*, eventos leídos, √s, año), cortes (con una frase de por qué cada
uno), *cutflow* en datos y MC, comparación datos/MC con el panel de cociente y *pile-up*, y una conclusión cuantitativa.
Presentas en 10 min sin leer las diapositivas. Prepárate para las preguntas de control del mes (al final del programa).